Import libraries

In [ ]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [ ]:
%time train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
%time test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

In [ ]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [ ]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [ ]:
print('X_train[:10]', X_train[:1].astype(float))

In [ ]:
print('y_train[:10]', y_train[:1].astype(float))

In [ ]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

In [ ]:
epochs = 150
alpha = 0.1
lr = 1e-3
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 30
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0
shared_dropout = 0.0
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

In [9]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_method = trial.suggest_categorical("method", ["mmd_rbf", "mmd_linear"])
    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence CFRNet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            cfr.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p= cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "method": t.params["method"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "method": t.params["method"],
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1874


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 38. Best value: 1.26157: 100%|██████████| 50/50 [1:09:30<00:00, 83.41s/it]


In [10]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 38
Best mean_Val_AUQC: 1.261568
Best params: {'lr': 8.96891731626277e-05, 'weight_decay': 1.0521745552369724e-05, 'alpha': 0.5, 'method': 'mmd_rbf'}

Best config table:


,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,0.00009,0.000011,0.5,mmd_rbf,1.261568,0.172405



Top 10 trials:


,trial,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,38,0.000090,0.000011,0.5,mmd_rbf,1.261568,0.172405
1,43,0.000097,0.000013,0.5,mmd_rbf,1.258789,0.164246
2,41,0.000100,0.000012,0.5,mmd_rbf,1.257921,0.169278
3,40,0.000104,0.000012,0.5,mmd_rbf,1.257450,0.164451
4,46,0.000097,0.000011,0.5,mmd_rbf,1.256604,0.166434
5,45,0.000096,0.000010,0.5,mmd_rbf,1.256293,0.167552
6,35,0.000087,0.000017,0.5,mmd_rbf,1.255343,0.173346
7,36,0.000084,0.000017,0.5,mmd_rbf,1.254805,0.165695
8,33,0.000085,0.000019,0.1,mmd_rbf,1.254475,0.170182
9,47,0.000103,0.000011,0.5,mmd_rbf,1.254187,0.164394


In [13]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [7,8,9,10,11]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfr = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=9.00E-05, 
        alpha = 0.5, method="mmd_rbf",
        weight_decay=1.10E-05, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_dropout=outcome_dropout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfr.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfr.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 7
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 0.0062 | Val Loss: 193.9340 | Val Qini: 1.1879 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.1879 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.0093 | Val Loss: 193.8902 | Val Qini: 1.1256 (patience: 1/20)EMA Trend: 1.1785 | (patience: 1/20)
Epoch 3/150 | Loss: 112.5429 | Val Loss: 193.8456 | Val Qini: 0.9856 (patience: 2/20)EMA Trend: 1.1496 | (patience: 2/20)
Epoch 4/150 | Loss: 0.0009 | Val Loss: 193.7974 | Val Qini: 0.8104 (patience: 3/20)EMA Trend: 1.0987 | (patience: 3/20)
Epoch 5/150 | Loss: 0.0030 | Val Loss: 193.7462 | Val Qini: 0.6813 (patience: 4/20)EMA Trend: 1.0361 | (patience: 4/20)
Epoch 6/150 | Loss: 697.0491 | Va

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.0199 | Val Loss: 193.7141 | Val Qini: -0.1747 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.1747 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.0466 | Val Loss: 193.6765 | Val Qini: -0.0411 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.1546 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.0050 | Val Loss: 193.6410 | Val Qini: 0.0925 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.1176 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.0239 | Val Loss: 193.6052 | Val Qini: 0.1451 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.0782 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.0198 | Val Loss: 193.5687 | Val Qini: 0.1478 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.0443 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.0238 | Val Loss: 193.5307 | Val Qini: 0.0516 ✓ above trend but not peak (patience: 1/20)EMA Trend: -0.0299 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: 0.0134 | Val Loss: 193.4900 | Val Qini: 0.0157 ✓ above trend but not peak (patience: 2/20)EMA Trend: -0.0231

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.0114 | Val Loss: 193.6684 | Val Qini: 1.4031 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.4031 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.0005 | Val Loss: 193.6250 | Val Qini: 1.4032 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.4031 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: -0.0026 | Val Loss: 193.5821 | Val Qini: 1.3495 (patience: 1/20)EMA Trend: 1.3951 | (patience: 1/20)
Epoch 4/150 | Loss: 0.0043 | Val Loss: 193.5371 | Val Qini: 1.3377 (patience: 2/20)EMA Trend: 1.3865 | (patience: 2/20)
Epoch 5/150 | Loss: 0.0062 | Val Loss: 193.4891 | Val Qini: 1.3389 (patience: 3/20)EMA Trend: 1.3793 | (patience: 3/20)
Epoch 6/150 | Loss: 0.0465 | Val Loss: 193.4373 | Val Qini: 1.3048 (patience: 4/20)EMA Trend: 1.3682 | (patience: 4/20)
Epoch 7/150 | Loss: 0.0433 | Val Loss: 193.3817 | Val Qini: 1.3056 (patience: 5/20)EMA Trend: 1.3588 | (patience: 5/20)
Epoch 8/150 | Loss: 0.0761 | Val Loss: 193.3241 | Val Qini: 1.3277 (patience: 6/20)EMA Trend: 1.3541 | (patience: 6/20)
E

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.0048 | Val Loss: 193.9865 | Val Qini: 1.2316 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.2316 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.0002 | Val Loss: 193.9410 | Val Qini: 1.2679 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.2371 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.0081 | Val Loss: 193.8935 | Val Qini: 1.3181 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.2492 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.0104 | Val Loss: 193.8441 | Val Qini: 1.3206 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.2599 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.0202 | Val Loss: 193.7926 | Val Qini: 1.3740 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.2770 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.0300 | Val Loss: 193.7371 | Val Qini: 1.4490 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.3028 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.0155 | Val Loss: 193.6776 | Val Qini: 1.4864 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.3304 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 0.0072 | Val 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.0181 | Val Loss: 193.7505 | Val Qini: 0.4608 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4608 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.0002 | Val Loss: 193.7276 | Val Qini: 0.6230 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4852 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.0237 | Val Loss: 193.7040 | Val Qini: 0.8190 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5352 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: -0.0155 | Val Loss: 193.6801 | Val Qini: 0.8857 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5878 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.0074 | Val Loss: 193.6542 | Val Qini: 0.9349 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6399 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.0014 | Val Loss: 193.6255 | Val Qini: 0.9452 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6857 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.0002 | Val Loss: 193.5930 | Val Qini: 0.9237 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.7214 | ✓ above trend but not peak (patience: 1/2

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
